In [1]:
# ============================================
# BM25 RETRIEVER (для BGE-LARGE-EN-V1.5)
# ============================================

from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import os
import json
import re
from time import time

# Скачиваем необходимые ресурсы
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


# -----------------------------
# Нормализация текста
# -----------------------------
def normalize_text(text: str) -> str:
    text = re.sub(r"[^\w\s]", " ", text)
    text = text.lower()
    return text


# -----------------------------
# Токенизация + лемматизация
# -----------------------------
def tokenize(text: str):
    text = normalize_text(text)
    tokens = nltk.word_tokenize(text)

    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]

    return tokens


# -----------------------------
# Загрузка чанков
# -----------------------------
def load_chunks(path):
    chunks = []
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            chunks.append(entry)
            texts.append(entry.get("text", ""))
    return chunks, texts


# -----------------------------
# Загрузка golden-сета
# -----------------------------
def load_queries(golden_path):
    queries = []
    with open(golden_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("query") or obj.get("question") or ""
            queries.append(q)
    return queries


# -----------------------------
# Построение BM25 индекса
# -----------------------------
def build_bm25_index(texts):
    tokenized = [tokenize(t) for t in texts]
    bm25 = BM25Okapi(tokenized)
    return bm25, tokenized


# -----------------------------
# BM25 поиск
# -----------------------------
def bm25_search(query, bm25, chunks, tokenized_texts, top_k=50):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []
    for idx, score in ranked:
        chunk = chunks[idx]
        results.append({
            "chunk_id": chunk.get("chunk_id"),
            "score": float(score)
        })
    return results


# -----------------------------
# Главная функция BM25
# -----------------------------
def run_bm25(
    chunks_file: str,
    golden_file: str,
    output_file: str,
    top_k: int = 50
):
    # ============================================================
    # НОВАЯ СТРУКТУРА ПАПОК ДЛЯ BGE-LARGE
    # ============================================================
    base_dir = "results_bge_large_en_v15"
    bm25_dir = os.path.join(base_dir, "results_bm25_bge_large")

    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
        print(f"[INFO] Created directory: {base_dir}")

    if not os.path.exists(bm25_dir):
        os.makedirs(bm25_dir)
        print(f"[INFO] Created directory: {bm25_dir}")

    # если передано только имя файла — кладём в results_bm25_bge_large/
    if "/" not in output_file and "\\" not in output_file:
        output_file = os.path.join(bm25_dir, output_file)

    print(f"[INFO] Output will be saved to: {output_file}")
    # ============================================================

    print(f"[INFO] Loading chunks from {chunks_file}...")
    chunks, texts = load_chunks(chunks_file)
    print(f"[INFO] Loaded {len(chunks)} chunks")

    print("[INFO] Building BM25 index...")
    t0 = time()
    bm25, tokenized = build_bm25_index(texts)
    print(f"[INFO] BM25 index ready (took {time()-t0:.1f}s)")

    print(f"[INFO] Loading queries from {golden_file}...")
    queries = load_queries(golden_file)
    print(f"[INFO] Loaded {len(queries)} queries")

    with open(output_file, "w", encoding="utf-8") as out:
        for i, q in enumerate(queries, start=1):
            if not q:
                out.write(json.dumps({"query": "", "retrieved": []}, ensure_ascii=False) + "\n")
                continue

            hits = bm25_search(q, bm25, chunks, tokenized, top_k=top_k)

            out.write(json.dumps({
                "query": q,
                "retrieved": hits
            }, ensure_ascii=False) + "\n")

            if i % 50 == 0 or i == len(queries):
                print(f"[PROGRESS] {i}/{len(queries)} queries processed")

    print(f"[OK] Results written to {output_file}")


In [2]:
run_bm25(
    chunks_file="chunks/chunks_fixed_60_overlap0.1.jsonl",
    golden_file="golden_sets/golden_fixed_60.jsonl",
    output_file="results_bm25_fixed_60_overlap0.1.jsonl",
    top_k=50
)

[INFO] Created directory: results_bge_large_en_v15
[INFO] Created directory: results_bge_large_en_v15\results_bm25_bge_large
[INFO] Output will be saved to: results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_60_overlap0.1.jsonl
[INFO] Loading chunks from chunks/chunks_fixed_60_overlap0.1.jsonl...
[INFO] Loaded 4785 chunks
[INFO] Building BM25 index...
[INFO] BM25 index ready (took 3.1s)
[INFO] Loading queries from golden_sets/golden_fixed_60.jsonl...
[INFO] Loaded 89 queries
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_60_overlap0.1.jsonl


In [3]:
from sentence_transformers import SentenceTransformer

MODEL_BGE = SentenceTransformer("BAAI/bge-large-en-v1.5")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ================================
# DENSE RETRIEVER (BGE-LARGE-EN-V1.5)
# ================================

import json
import numpy as np
from time import time


# --------------------------------
# Загрузка чанков
# --------------------------------
def load_chunks(path):
    chunks = []
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            chunks.append(entry)
            texts.append(entry.get("text", ""))
    return chunks, texts


# --------------------------------
# Загрузка golden-сета
# --------------------------------
def load_queries(golden_path):
    queries = []
    with open(golden_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("query") or obj.get("question") or ""
            queries.append(q)
    return queries


# --------------------------------
# Построение dense-индекса
# --------------------------------
def build_dense_index(texts, model):
    """
    Для BGE:
    - НЕ используем префиксы "passage:" или "query:"
    - обязательно normalize_embeddings=True
    """
    print("[INFO] Encoding chunk embeddings...")
    t0 = time()

    chunk_embs = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )

    print(f"[INFO] Chunk embeddings ready (took {time()-t0:.1f}s)")
    return np.array(chunk_embs)


# --------------------------------
# Dense поиск
# --------------------------------
def dense_search(query, model, chunk_embs, chunks, top_k=50):
    query_emb = model.encode(
        query,
        normalize_embeddings=True
    )

    scores = np.dot(chunk_embs, query_emb)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        chunk = chunks[idx]
        results.append({
            "chunk_id": chunk.get("chunk_id"),
            "score": float(scores[idx])
        })

    return results


# --------------------------------
# Главная функция dense-retriever
# --------------------------------
def run_dense(
    chunks_file: str,
    golden_file: str,
    output_file: str,
    top_k: int = 50
):
    # ============================================================
    # НОВАЯ СТРУКТУРА ПАПОК ДЛЯ BGE-LARGE
    # ============================================================
    base_dir = "results_bge_large_en_v15"
    dense_dir = os.path.join(base_dir, "results_dense_bge_large")

    # создаём папки, если их нет
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
        print(f"[INFO] Created directory: {base_dir}")

    if not os.path.exists(dense_dir):
        os.makedirs(dense_dir)
        print(f"[INFO] Created directory: {dense_dir}")

    
    if "/" not in output_file and "\\" not in output_file:
        output_file = os.path.join(dense_dir, output_file)

    print(f"[INFO] Output will be saved to: {output_file}")
    # ============================================================

    print(f"[INFO] Loading chunks from {chunks_file}...")
    chunks, texts = load_chunks(chunks_file)
    print(f"[INFO] Loaded {len(chunks)} chunks")

    print("[INFO] Loading BGE-LARGE-EN-V1.5 model...")
    model = MODEL_BGE   # <-- новая модель

    # строим dense-индекс
    chunk_embs = build_dense_index(texts, model)

    print(f"[INFO] Loading queries from {golden_file}...")
    queries = load_queries(golden_file)
    print(f"[INFO] Loaded {len(queries)} queries")

    print("[INFO] Running dense retrieval...")

    with open(output_file, "w", encoding="utf-8") as out:
        for i, q in enumerate(queries, start=1):
            if not q:
                out.write(json.dumps({"query": "", "retrieved": []}, ensure_ascii=False) + "\n")
                continue

            hits = dense_search(q, model, chunk_embs, chunks, top_k=top_k)

            out.write(json.dumps({
                "query": q,
                "retrieved": hits
            }, ensure_ascii=False) + "\n")

            if i % 50 == 0 or i == len(queries):
                print(f"[PROGRESS] {i}/{len(queries)} queries processed")

    print(f"[OK] Results written to {output_file}")


In [5]:
run_dense(
    chunks_file="chunks/chunks_fixed_60_overlap0.1.jsonl",
    golden_file="golden_sets/golden_fixed_60.jsonl",
    output_file="results_dense_fixed_60_overlap0.1.jsonl",
    top_k=50
)


[INFO] Created directory: results_bge_large_en_v15\results_dense_bge_large
[INFO] Output will be saved to: results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_60_overlap0.1.jsonl
[INFO] Loading chunks from chunks/chunks_fixed_60_overlap0.1.jsonl...
[INFO] Loaded 4785 chunks
[INFO] Loading BGE-LARGE-EN-V1.5 model...
[INFO] Encoding chunk embeddings...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 55.6s)
[INFO] Loading queries from golden_sets/golden_fixed_60.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_60_overlap0.1.jsonl


In [6]:
# ================================
# HYBRID RETRIEVER (RRF) — BGE-LARGE-EN-V1.5
# ================================

import json
import os


# --------------------------------
# Загрузка результатов BM25/Dense
# --------------------------------
def load_results(path):
    results = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            results.append(json.loads(line))
    return results


# --------------------------------
# RRF Fusion
# --------------------------------
def rrf_fusion(bm25_hits, dense_hits, k=60, top_k=50):
    scores = {}

    # BM25 ранги
    for rank, item in enumerate(bm25_hits, start=1):
        cid = item["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank)

    # Dense ранги
    for rank, item in enumerate(dense_hits, start=1):
        cid = item["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (k + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [
        {"chunk_id": cid, "score": float(score)}
        for cid, score in ranked[:top_k]
    ]


# --------------------------------
# Главная функция гибрида
# --------------------------------
def run_hybrid(
    bm25_file: str,
    dense_file: str,
    output_file: str,
    top_k: int = 50
):
    # ============================================================
    # НОВАЯ СТРУКТУРА ПАПОК ДЛЯ BGE-LARGE
    # ============================================================
    base_dir = "results_bge_large_en_v15"
    hybrid_dir = os.path.join(base_dir, "results_hybrid_bge_large")

    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
        print(f"[INFO] Created directory: {base_dir}")

    if not os.path.exists(hybrid_dir):
        os.makedirs(hybrid_dir)
        print(f"[INFO] Created directory: {hybrid_dir}")

    # если передано только имя файла — кладём в results_hybrid_bge_large/
    if "/" not in output_file and "\\" not in output_file:
        output_file = os.path.join(hybrid_dir, output_file)

    print(f"[INFO] Output will be saved to: {output_file}")
    # ============================================================

    print(f"[INFO] Loading BM25 results from {bm25_file}...")
    bm25_results = load_results(bm25_file)
    print(f"[INFO] Loaded {len(bm25_results)} BM25 queries")

    print(f"[INFO] Loading Dense results from {dense_file}...")
    dense_results = load_results(dense_file)
    print(f"[INFO] Loaded {len(dense_results)} Dense queries")

    assert len(bm25_results) == len(dense_results), \
        "BM25 и Dense должны иметь одинаковое количество запросов!"

    print("[INFO] Running RRF fusion...")

    with open(output_file, "w", encoding="utf-8") as out:
        for i, (bm25_row, dense_row) in enumerate(zip(bm25_results, dense_results), start=1):

            q = bm25_row["query"]
            bm25_hits = bm25_row["retrieved"]
            dense_hits = dense_row["retrieved"]

            fused = rrf_fusion(bm25_hits, dense_hits, top_k=top_k)

            out.write(json.dumps({
                "query": q,
                "retrieved": fused
            }, ensure_ascii=False) + "\n")

            if i % 50 == 0 or i == len(bm25_results):
                print(f"[PROGRESS] {i}/{len(bm25_results)} queries processed")

    print(f"[OK] Hybrid results written to {output_file}")


In [10]:
run_hybrid(
    bm25_file="results_bge_large_en_v15/results_bm25_bge_large/results_bm25_fixed_60_overlap0.1.jsonl",
    dense_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_60_overlap0.1.jsonl",
    output_file="results_hybrid_fixed_60_overlap0.1.jsonl",
    top_k=50
)


[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_fixed_60_overlap0.1.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15/results_bm25_bge_large/results_bm25_fixed_60_overlap0.1.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_60_overlap0.1.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Hybrid results written to results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_fixed_60_overlap0.1.jsonl


In [11]:
import math

def evaluate_all_metrics(golden_file, bm25_file, dense_file, hybrid_file, k=5):
    """
    Полностью автономная версия:
    - загружает golden-set
    - загружает результаты BM25/Dense/Hybrid
    - считает Recall@k, MRR, nDCG@k
    """

    # -----------------------------
    # 1. Загрузка golden-set
    # -----------------------------
    golden = {}
    with open(golden_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            golden[obj["query"]] = set(obj["gold_chunks"])

    # -----------------------------
    # 2. Загрузка результатов
    # -----------------------------
    def load_results(path):
        results = {}
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                q = obj["query"]
                retrieved = [x["chunk_id"] for x in obj["retrieved"]]
                results[q] = retrieved
        return results

    bm25_map = load_results(bm25_file)
    dense_map = load_results(dense_file)
    hybrid_map = load_results(hybrid_file)

    # -----------------------------
    # 3. Метрики
    # -----------------------------
    def recall_at_k(retrieved, gold, k):
        if not gold:
            return 0.0
        return 1.0 if any(r in gold for r in retrieved[:k]) else 0.0

    def mrr(retrieved, gold):
        for rank, cid in enumerate(retrieved, start=1):
            if cid in gold:
                return 1.0 / rank
        return 0.0

    def ndcg_at_k(retrieved, gold, k):
        dcg = 0.0
        for i, cid in enumerate(retrieved[:k]):
            if cid in gold:
                dcg += 1.0 / (math.log2(i + 2))
        # ideal DCG
        ideal_hits = min(len(gold), k)
        idcg = sum(1.0 / (math.log2(i + 2)) for i in range(ideal_hits))
        return dcg / idcg if idcg > 0 else 0.0

    # -----------------------------
    # 4. Оценка одного метода
    # -----------------------------
    def evaluate_method(results_map):
        recalls = []
        mrrs = []
        ndcgs = []

        for q, gold_chunks in golden.items():
            retrieved = results_map.get(q, [])

            recalls.append(recall_at_k(retrieved, gold_chunks, k))
            mrrs.append(mrr(retrieved, gold_chunks))
            ndcgs.append(ndcg_at_k(retrieved, gold_chunks, k))

        return {
            f"Recall@{k}": sum(recalls) / len(recalls),
            "MRR": sum(mrrs) / len(mrrs),
            f"nDCG@{k}": sum(ndcgs) / len(ndcgs)
        }

    # -----------------------------
    # 5. Итог
    # -----------------------------
    return {
        "bm25": evaluate_method(bm25_map),
        "dense": evaluate_method(dense_map),
        "hybrid": evaluate_method(hybrid_map)
    }


In [12]:
metrics = evaluate_all_metrics(
    golden_file="golden_sets/golden_fixed_60.jsonl",
    bm25_file="results_bge_large_en_v15/results_bm25_bge_large/results_bm25_fixed_60_overlap0.1.jsonl",
    dense_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_60_overlap0.1.jsonl",
    hybrid_file="results_bge_large_en_v15/results_hybrid_bge_large/results_hybrid_fixed_60_overlap0.1.jsonl",
    k=5
)

metrics



{'bm25': {'Recall@5': 0.7528089887640449,
  'MRR': 0.594738236891752,
  'nDCG@5': 0.3704457300965113},
 'dense': {'Recall@5': 0.797752808988764,
  'MRR': 0.6205579518634283,
  'nDCG@5': 0.40187996200692133},
 'hybrid': {'Recall@5': 0.797752808988764,
  'MRR': 0.5990212783471215,
  'nDCG@5': 0.4052061736067329}}

In [13]:
import os
import json
from pathlib import Path
import math


# ============================================================
# ЕДИНЫЙ ПАЙПЛАЙН ДЛЯ BGE-LARGE-EN-V1.5 (без пересоздания чанков)
# ============================================================

def run_full_pipeline(
    chunk_size: int,
    overlap_ratio: float,
    mode: str,
    base_dir: str = ".",
    top_k: int = 5
):
    print("\n==============================")
    print(f"  RUN PIPELINE: {mode}, size={chunk_size}, overlap={overlap_ratio}")
    print("==============================\n")

    # ---------------------------------------------------------
    # Папки (НОВАЯ СТРУКТУРА)
    # ---------------------------------------------------------
    chunks_dir = Path(base_dir) / "chunks"
    golden_dir = Path(base_dir) / "golden_sets"

    results_root = Path(base_dir) / "results_bge_large_en_v15"
    bm25_dir = results_root / "results_bm25_bge_large"
    dense_dir = results_root / "results_dense_bge_large"
    hybrid_dir = results_root / "results_hybrid_bge_large"

    metrics_root = Path(base_dir) / "metrics_bge_large_en_v_poltora"

    # создаём папки
    bm25_dir.mkdir(parents=True, exist_ok=True)
    dense_dir.mkdir(parents=True, exist_ok=True)
    hybrid_dir.mkdir(parents=True, exist_ok=True)
    metrics_root.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------
    # 1. Используем существующие ЧАНКИ
    # ---------------------------------------------------------
    chunks_file = chunks_dir / f"chunks_{mode}_{chunk_size}_overlap{overlap_ratio}.jsonl"

    if not chunks_file.exists():
        raise FileNotFoundError(f"Файл чанков не найден: {chunks_file}")

    print(f"[OK] Using existing chunks: {chunks_file}\n")

    # ---------------------------------------------------------
    # 2. Используем существующий GOLDEN SET
    # ---------------------------------------------------------
    golden_file = golden_dir / f"golden_{mode}_{chunk_size}.jsonl"

    if not golden_file.exists():
        raise FileNotFoundError(f"Golden set не найден: {golden_file}")

    print(f"[OK] Using existing golden set: {golden_file}\n")

    # ---------------------------------------------------------
    # 3. BM25
    # ---------------------------------------------------------
    bm25_file = bm25_dir / f"results_bm25_{mode}_{chunk_size}_overlap{overlap_ratio}.jsonl"

    run_bm25(
        chunks_file=str(chunks_file),
        golden_file=str(golden_file),
        output_file=str(bm25_file),
        top_k=top_k
    )

    print(f"[OK] BM25 results: {bm25_file}\n")

    # ---------------------------------------------------------
    # 4. DENSE
    # ---------------------------------------------------------
    dense_file = dense_dir / f"results_dense_{mode}_{chunk_size}_overlap{overlap_ratio}.jsonl"

    run_dense(
        chunks_file=str(chunks_file),
        golden_file=str(golden_file),
        output_file=str(dense_file),
        top_k=top_k
    )

    print(f"[OK] Dense results: {dense_file}\n")

    # ---------------------------------------------------------
    # 5. HYBRID
    # ---------------------------------------------------------
    hybrid_file = hybrid_dir / f"results_hybrid_{mode}_{chunk_size}_overlap{overlap_ratio}.jsonl"

    run_hybrid(
        bm25_file=str(bm25_file),
        dense_file=str(dense_file),
        output_file=str(hybrid_file),
        top_k=top_k
    )

    print(f"[OK] Hybrid results: {hybrid_file}\n")

    # ---------------------------------------------------------
    # 6. МЕТРИКИ
    # ---------------------------------------------------------
    metrics = evaluate_all_metrics(
        golden_file=str(golden_file),
        bm25_file=str(bm25_file),
        dense_file=str(dense_file),
        hybrid_file=str(hybrid_file),
        k=top_k
    )

    metrics_file = metrics_root / f"metrics_{mode}_{chunk_size}_overlap{overlap_ratio}.json"

    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"[OK] Metrics saved to: {metrics_file}\n")

    return {
        "chunks": str(chunks_file),
        "golden": str(golden_file),
        "bm25": str(bm25_file),
        "dense": str(dense_file),
        "hybrid": str(hybrid_file),
        "metrics": str(metrics_file)
    }


In [14]:
chunk_sizes = [50, 100, 150, 200, 250, 300, 400, 600]
overlaps = [0.1, 0.2, 0.3]
modes = ["fixed", "structured"]

all_results = []

for mode in modes:
    for size in chunk_sizes:
        for ov in overlaps:
            print(f"\n=== RUN {mode} size={size} overlap={ov} ===\n")
            res = run_full_pipeline(
                chunk_size=size,
                overlap_ratio=ov,
                mode=mode,
                base_dir=".",
                top_k=5
            )
            all_results.append(res)



=== RUN fixed size=50 overlap=0.1 ===


  RUN PIPELINE: fixed, size=50, overlap=0.1

[OK] Using existing chunks: chunks\chunks_fixed_50_overlap0.1.jsonl

[OK] Using existing golden set: golden_sets\golden_fixed_50.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_50_overlap0.1.jsonl
[INFO] Loading chunks from chunks\chunks_fixed_50_overlap0.1.jsonl...
[INFO] Loaded 5735 chunks
[INFO] Building BM25 index...
[INFO] BM25 index ready (took 0.9s)
[INFO] Loading queries from golden_sets\golden_fixed_50.jsonl...
[INFO] Loaded 89 queries
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_50_overlap0.1.jsonl
[OK] BM25 results: results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_50_overlap0.1.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.1.json

Batches:   0%|          | 0/180 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 57.4s)
[INFO] Loading queries from golden_sets\golden_fixed_50.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.1.jsonl
[OK] Dense results: results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.1.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_fixed_50_overlap0.1.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_50_overlap0.1.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.1.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Hyb

Batches:   0%|          | 0/202 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 66.0s)
[INFO] Loading queries from golden_sets\golden_fixed_50.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.2.jsonl
[OK] Dense results: results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.2.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_fixed_50_overlap0.2.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15\results_bm25_bge_large\results_bm25_fixed_50_overlap0.2.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15\results_dense_bge_large\results_dense_fixed_50_overlap0.2.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Hyb

...

[INFO] Chunk embeddings ready (took 69.4s)
[INFO] Loading queries from golden_sets\golden_structured_600.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.1.jsonl
[OK] Dense results: results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.1.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_structured_600_overlap0.1.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15\results_bm25_bge_large\results_bm25_structured_600_overlap0.1.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.1.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRE

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 71.4s)
[INFO] Loading queries from golden_sets\golden_structured_600.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.2.jsonl
[OK] Dense results: results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.2.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_structured_600_overlap0.2.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15\results_bm25_bge_large\results_bm25_structured_600_overlap0.2.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.2.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRE

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 75.6s)
[INFO] Loading queries from golden_sets\golden_structured_600.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.3.jsonl
[OK] Dense results: results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.3.jsonl

[INFO] Output will be saved to: results_bge_large_en_v15\results_hybrid_bge_large\results_hybrid_structured_600_overlap0.3.jsonl
[INFO] Loading BM25 results from results_bge_large_en_v15\results_bm25_bge_large\results_bm25_structured_600_overlap0.3.jsonl...
[INFO] Loaded 89 BM25 queries
[INFO] Loading Dense results from results_bge_large_en_v15\results_dense_bge_large\results_dense_structured_600_overlap0.3.jsonl...
[INFO] Loaded 89 Dense queries
[INFO] Running RRF fusion...
[PROGRESS] 50/89 queries processed
[PROGRE

In [15]:
import pandas as pd

metrics_dir = Path("metrics_bge_large_en_v_poltora")

rows = []

pattern = re.compile(r"metrics_(fixed|structured)_(\d+)_overlap([0-9.]+)\.json")

for file in metrics_dir.glob("metrics_*.json"):
    m = pattern.match(file.name)
    if not m:
        continue
    
    mode = m.group(1)
    chunk_size = int(m.group(2))
    overlap = float(m.group(3))

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for retriever in ["bm25", "dense", "hybrid"]:
        rows.append({
            "mode": mode,
            "chunk_size": chunk_size,
            "overlap": overlap,
            "retriever": retriever,
            "Recall@5": data[retriever]["Recall@5"],
            "MRR": data[retriever]["MRR"],
            "nDCG@5": data[retriever]["nDCG@5"],
        })

df = pd.DataFrame(rows)
df.head(100)


,mode,chunk_size,overlap,retriever,Recall@5,MRR,nDCG@5
0,fixed,100,0.1,bm25,0.752809,0.580337,0.386404
1,fixed,100,0.1,dense,0.797753,0.657303,0.406952
2,fixed,100,0.1,hybrid,0.831461,0.644569,0.410866
3,fixed,100,0.2,bm25,0.730337,0.579963,0.388115
4,fixed,100,0.2,dense,0.808989,0.657678,0.424616
...,...,...,...,...,...,...,...
95,structured,200,0.2,hybrid,0.752809,0.579963,0.355650
96,structured,200,0.3,bm25,0.674157,0.526030,0.327635
97,structured,200,0.3,dense,0.775281,0.575094,0.379393
98,structured,200,0.3,hybrid,0.775281,0.592135,0.373442


In [16]:
# Будем ранжировать численно 
df["score"] = (
    0.5 * df["Recall@5"] +
    0.3 * df["nDCG@5"] +
    0.2 * df["MRR"]
)

df_sorted = df.sort_values("score", ascending=False)

df_sorted.head(10)

,mode,chunk_size,overlap,retriever,Recall@5,MRR,nDCG@5,score
52,fixed,400,0.3,dense,0.876404,0.729213,0.501161,0.734393
49,fixed,400,0.2,dense,0.853933,0.720225,0.473312,0.713005
58,fixed,50,0.2,dense,0.887640,0.670974,0.439349,0.709820
37,fixed,300,0.1,dense,0.876404,0.686517,0.434200,0.705766
13,fixed,150,0.2,dense,0.853933,0.731461,0.434649,0.703653
53,fixed,400,0.3,hybrid,0.842697,0.688202,0.479645,0.702882
31,fixed,250,0.2,dense,0.853933,0.702060,0.442356,0.700085
64,fixed,600,0.1,dense,0.842697,0.689888,0.464433,0.698656
68,fixed,600,0.2,hybrid,0.853933,0.668165,0.455680,0.697303
50,fixed,400,0.2,hybrid,0.842697,0.678839,0.461603,0.695597


### Сравниваем bi-encoder и cross-encoder ###
Возьмем лучшую стратегию за исключеним по 600...
И добавим реранкер которому будем давать разные вариации количества кандидатов найденных Dense 
И сравним все полученное между собой и с dense который искал чанки через be-encoder

In [44]:
run_dense(
    chunks_file="chunks/chunks_fixed_400_overlap0.3.jsonl",
    golden_file="golden_sets/golden_fixed_400.jsonl",
    output_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top150.jsonl",
    top_k=150
)


[INFO] Output will be saved to: results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top150.jsonl
[INFO] Loading chunks from chunks/chunks_fixed_400_overlap0.3.jsonl...
[INFO] Loaded 993 chunks
[INFO] Loading BGE-LARGE-EN-V1.5 model...
[INFO] Encoding chunk embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[INFO] Chunk embeddings ready (took 207.3s)
[INFO] Loading queries from golden_sets/golden_fixed_400.jsonl...
[INFO] Loaded 89 queries
[INFO] Running dense retrieval...
[PROGRESS] 50/89 queries processed
[PROGRESS] 89/89 queries processed
[OK] Results written to results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top150.jsonl


In [40]:
# ============================================================
# CROSS-ENCODER RERANKER — BGE-RERANKER-BASE
# ============================================================

RERANKER_NAME = "models/bge-reranker-base"

BGE_TOKENIZER = AutoTokenizer.from_pretrained(RERANKER_NAME, local_files_only=True)
BGE_MODEL = AutoModelForSequenceClassification.from_pretrained(RERANKER_NAME, local_files_only=True).cuda().eval()



def run_reranker_bge(
    chunks_file: str,
    dense_file: str,
    output_file: str,
    top_k: int = 50,
    batch_size: int = 16
):
    """
    Reranking top_k результатов dense-ретривера с помощью BGE-RERANKER-BASE.
    """

    tokenizer = BGE_TOKENIZER
    model = BGE_MODEL

    # 1. Загружаем чанки
    chunk_map = {}
    with open(chunks_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            chunk_map[obj["chunk_id"]] = obj["text"]

    # 2. Загружаем dense результаты
    dense_results = []
    with open(dense_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            dense_results.append(json.loads(line))

    # 3. Готовим выходную папку
    out_dir = os.path.dirname(output_file)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # 4. Reranking
    with open(output_file, "w", encoding="utf-8") as out:

        for row in tqdm(dense_results, desc=f"Reranking BGE top_k={top_k}"):
            query = row["query"]
            candidates = row["retrieved"][:top_k]

            pairs = [
                (query, chunk_map[c["chunk_id"]])
                for c in candidates
                if c["chunk_id"] in chunk_map
            ]

            scores = []

            for i in range(0, len(pairs), batch_size):
                batch = pairs[i:i+batch_size]

                enc = tokenizer(
                    [q for q, p in batch],
                    [p for q, p in batch],
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                ).to(model.device)

                with torch.no_grad():
                    logits = model(**enc).logits.squeeze(-1)

                scores.extend(logits.cpu().tolist())

            reranked = sorted(
                zip(candidates, scores),
                key=lambda x: x[1],
                reverse=True
            )

            out.write(json.dumps({
                "query": query,
                "retrieved": [
                    {"chunk_id": c["chunk_id"], "score": float(s)}
                    for c, s in reranked
                ]
            }, ensure_ascii=False) + "\n")

    print(f"[OK] BGE-Reranker results saved to: {output_file}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: models/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
def evaluate_reranker_vs_dense(
    golden_file,
    dense_file,
    reranker_file,
    k=5,
    output_dir="metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker"
):
    """
    Сравнение dense vs BGE cross-reranker по метрикам.
    """

    import json, os, math

    # 1. Загрузка golden-set
    golden = {}
    with open(golden_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            golden[obj["query"]] = set(obj["gold_chunks"])

    # 2. Загрузка результатов
    def load_results(path):
        results = {}
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                q = obj["query"]
                retrieved = [x["chunk_id"] for x in obj["retrieved"]]
                results[q] = retrieved
        return results

    dense_map = load_results(dense_file)
    reranker_map = load_results(reranker_file)

    # 3. Метрики
    def recall_at_k(retrieved, gold, k):
        if not gold:
            return 0.0
        return 1.0 if any(r in gold for r in retrieved[:k]) else 0.0

    def mrr(retrieved, gold):
        for rank, cid in enumerate(retrieved, start=1):
            if cid in gold:
                return 1.0 / rank
        return 0.0

    def ndcg_at_k(retrieved, gold, k):
        dcg = 0.0
        for i, cid in enumerate(retrieved[:k]):
            if cid in gold:
                dcg += 1.0 / (math.log2(i + 2))
        ideal_hits = min(len(gold), k)
        idcg = sum(1.0 / (math.log2(i + 2)) for i in range(ideal_hits))
        return dcg / idcg if idcg > 0 else 0.0

    def evaluate_method(results_map):
        recalls = []
        mrrs = []
        ndcgs = []

        for q, gold_chunks in golden.items():
            retrieved = results_map.get(q, [])
            recalls.append(recall_at_k(retrieved, gold_chunks, k))
            mrrs.append(mrr(retrieved, gold_chunks))
            ndcgs.append(ndcg_at_k(retrieved, gold_chunks, k))

        return {
            f"Recall@{k}": sum(recalls) / len(recalls),
            "MRR": sum(mrrs) / len(mrrs),
            f"nDCG@{k}": sum(ndcgs) / len(ndcgs)
        }

    result = {
        "dense_file": dense_file,
        "reranker_file": reranker_file,
        "k": k,
        "dense": evaluate_method(dense_map),
        "reranker": evaluate_method(reranker_map)
    }

    os.makedirs(output_dir, exist_ok=True)

    dense_name = os.path.basename(dense_file).replace(".jsonl", "")
    rerank_name = os.path.basename(reranker_file).replace(".jsonl", "")

    out_path = os.path.join(
        output_dir,
        f"metrics_{dense_name}__vs__{rerank_name}__k{k}.json"
    )

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=4)

    print(f"[OK] BGE cross-enc metrics saved to: {out_path}")

    return result


In [50]:
top_k_values = [50, 100, 150, 200, 250, 300]

for top_k in top_k_values:
    print(f"\n=== BGE-Reranker top_k={top_k} ===")

    reranker_file = f"results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k{top_k}.jsonl"

    run_reranker_bge(
        chunks_file="chunks/chunks_fixed_400_overlap0.3.jsonl",
        dense_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top300.jsonl",
        output_file=reranker_file,
        top_k=top_k
    )

    evaluate_reranker_vs_dense(
        golden_file="golden_sets/golden_fixed_400.jsonl",
        dense_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top300.jsonl",
        reranker_file=reranker_file,
        k=5
    )
    



=== BGE-Reranker top_k=50 ===


Reranking BGE top_k=50: 100%|██████████| 89/89 [47:35<00:00, 32.08s/it]


[OK] BGE-Reranker results saved to: results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k50.jsonl
[OK] BGE cross-enc metrics saved to: metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker\metrics_results_dense_fixed_400_overlap0.3_top300__vs__results_reranker_bge_k50__k5.json

=== BGE-Reranker top_k=100 ===


Reranking BGE top_k=100: 100%|██████████| 89/89 [1:37:58<00:00, 66.05s/it]


[OK] BGE-Reranker results saved to: results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k100.jsonl
[OK] BGE cross-enc metrics saved to: metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker\metrics_results_dense_fixed_400_overlap0.3_top300__vs__results_reranker_bge_k100__k5.json

=== BGE-Reranker top_k=150 ===


...

[OK] BGE-Reranker results saved to: results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k200.jsonl
[OK] BGE cross-enc metrics saved to: metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker\metrics_results_dense_fixed_400_overlap0.3_top300__vs__results_reranker_bge_k200__k5.json

=== BGE-Reranker top_k=250 ===


Reranking BGE top_k=250: 100%|██████████| 89/89 [4:06:51<00:00, 166.42s/it]  


[OK] BGE-Reranker results saved to: results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k250.jsonl
[OK] BGE cross-enc metrics saved to: metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker\metrics_results_dense_fixed_400_overlap0.3_top300__vs__results_reranker_bge_k250__k5.json

=== BGE-Reranker top_k=300 ===


Reranking BGE top_k=300: 100%|██████████| 89/89 [4:42:12<00:00, 190.26s/it]  


[OK] BGE-Reranker results saved to: results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k300.jsonl
[OK] BGE cross-enc metrics saved to: metrics_bge_large_en_v_poltora/metrics_dense_vs_cross_enc/metrics_cross_bge_reranker\metrics_results_dense_fixed_400_overlap0.3_top300__vs__results_reranker_bge_k300__k5.json


In [49]:


def load_results(path):
    results = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj["query"]
            retrieved = [x["chunk_id"] for x in obj["retrieved"]]
            results[q] = retrieved
    return results


def recall_at_k(retrieved, gold, k):
    return 1.0 if any(r in gold for r in retrieved[:k]) else 0.0


def mrr(retrieved, gold):
    for rank, cid in enumerate(retrieved, start=1):
        if cid in gold:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved, gold, k):
    dcg = 0.0
    for i, cid in enumerate(retrieved[:k]):
        if cid in gold:
            dcg += 1.0 / (math.log2(i + 2))
    ideal_hits = min(len(gold), k)
    idcg = sum(1.0 / (math.log2(i + 2)) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def compare_dense_vs_reranker(
    golden_file,
    dense_file,
    reranker_file,
    k=5
):
    # load golden
    golden = {}
    with open(golden_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            golden[obj["query"]] = set(obj["gold_chunks"])

    dense_map = load_results(dense_file)
    reranker_map = load_results(reranker_file)

    metrics = {
        "Method": [],
        f"Recall@{k}": [],
        "MRR": [],
        f"nDCG@{k}": []
    }

    for name, results_map in [
        ("Dense", dense_map),
        ("Reranker", reranker_map)
    ]:
        recalls = []
        mrrs = []
        ndcgs = []

        for q, gold_chunks in golden.items():
            retrieved = results_map.get(q, [])
            recalls.append(recall_at_k(retrieved, gold_chunks, k))
            mrrs.append(mrr(retrieved, gold_chunks))
            ndcgs.append(ndcg_at_k(retrieved, gold_chunks, k))

        metrics["Method"].append(name)
        metrics[f"Recall@{k}"].append(sum(recalls) / len(recalls))
        metrics["MRR"].append(sum(mrrs) / len(mrrs))
        metrics[f"nDCG@{k}"].append(sum(ndcgs) / len(ndcgs))

    return pd.DataFrame(metrics)


# === RUN COMPARISON ===

df = compare_dense_vs_reranker(
    golden_file="golden_sets/golden_fixed_400.jsonl",
    dense_file="results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_400_overlap0.3_top300.jsonl",
    reranker_file="results_bge_large_en_v15/results_reranker_bge/results_reranker_bge_k100.jsonl",
    k=5
)

df


,Method,Recall@5,MRR,nDCG@5
0,Dense,0.876404,0.740692,0.501161
1,Reranker,0.853933,0.755799,0.473060


In [1]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Tokenizer loaded!")


c:\Users\suslo\Desktop\ВКР\VKR\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded!


In [6]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",        # автоматически использует GPU, если есть
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Model loaded!")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:15<00:00, 27.75it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Model loaded!


In [ ]:
import os
import json
from glob import glob

METRICS_DIR = "metrics_bge_large_en_v_poltora"

def compute_integral_score(d):
    return 0.5 * d["Recall@5"] + 0.3 * d["nDCG@5"] + 0.2 * d["MRR"]

results = []

for path in glob(os.path.join(METRICS_DIR, "metrics_*.json")):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # имя стратегии = имя файла без расширения
    filename = os.path.basename(path).replace(".json", "")

    for retriever_name, metrics in data.items():
        score = compute_integral_score(metrics)

        results.append({
            "strategy": filename,          # fixed_100_overlap0.2
            "retriever": retriever_name,   # bm25 / dense / hybrid
            "score": score,
            "Recall@5": metrics["Recall@5"],
            "MRR": metrics["MRR"],
            "nDCG@5": metrics["nDCG@5"]
        })

# сортировка по убыванию интегрального скора
results_sorted = sorted(results, key=lambda x: x["score"], reverse=True)

print("=== RANKED STRATEGIES (BEST FIRST) ===\n")
for r in results_sorted:
    print(
        f"{r['strategy']:35s} | "
        f"{r['retriever']:7s} | "
        f"Score={r['score']:.4f} | "
        f"R@5={r['Recall@5']:.3f} | "
        f"MRR={r['MRR']:.3f} | "
        f"nDCG@5={r['nDCG@5']:.3f}"
    )


=== RANKED STRATEGIES (BEST FIRST) ===

metrics_fixed_400_overlap0.3        | dense   | Score=0.7344 | R@5=0.876 | MRR=0.729 | nDCG@5=0.501
metrics_fixed_400_overlap0.2        | dense   | Score=0.7130 | R@5=0.854 | MRR=0.720 | nDCG@5=0.473
metrics_fixed_50_overlap0.2         | dense   | Score=0.7098 | R@5=0.888 | MRR=0.671 | nDCG@5=0.439
metrics_fixed_300_overlap0.1        | dense   | Score=0.7058 | R@5=0.876 | MRR=0.687 | nDCG@5=0.434
metrics_fixed_150_overlap0.2        | dense   | Score=0.7037 | R@5=0.854 | MRR=0.731 | nDCG@5=0.435
metrics_fixed_400_overlap0.3        | hybrid  | Score=0.7029 | R@5=0.843 | MRR=0.688 | nDCG@5=0.480
metrics_fixed_250_overlap0.2        | dense   | Score=0.7001 | R@5=0.854 | MRR=0.702 | nDCG@5=0.442
metrics_fixed_600_overlap0.1        | dense   | Score=0.6987 | R@5=0.843 | MRR=0.690 | nDCG@5=0.464
metrics_fixed_600_overlap0.2        | hybrid  | Score=0.6973 | R@5=0.854 | MRR=0.668 | nDCG@5=0.456
metrics_fixed_400_overlap0.2        | hybrid  | Score=0.6956

### Пачка ###

In [ ]:
# Ячейка: generate_answers_from_retrievals.py
import json
import os
from tqdm import tqdm
import re

# ==========================
# 1. Очистка контекста
# ==========================
def clean_context(text):
    # Удаляем опасные токены Qwen
    text = text.replace("<|im_start|>", "")
    text = text.replace("<|im_end|>", "")
    text = text.replace("User:", "")
    text = text.replace("Assistant:", "")
    text = text.replace("Human:", "")
    text = text.replace("###", "")

    # Удаляем кодовые блоки
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)

    # Удаляем HTML-теги
    text = re.sub(r"<[^>]+>", " ", text)

    # Удаляем markdown-картинки
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)

    # Удаляем markdown-ссылки
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Удаляем таблицы
    text = "\n".join(
        line for line in text.splitlines()
        if not re.match(r"^\s*\|.*\|\s*$", line)
    )

    # Убираем лишние пробелы
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ==========================
# 2. Загрузка чанков
# ==========================
def load_chunks(chunks_path):
    chunks = {}
    with open(chunks_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            cid = item.get("chunk_id") or item.get("id") or item.get("id_chunk")
            if not cid:
                continue
            chunks[cid] = item.get("text", "")
    return chunks


# ==========================
# 3. Построение контекста
# ==========================
def build_context_from_retrieved(retrieved_list, chunks_map, top_k=5, max_chars=None):
    ctx_parts = []
    for r in retrieved_list[:top_k]:
        cid = r.get("chunk_id")
        text = chunks_map.get(cid)
        if text:
            ctx_parts.append(f"--- CHUNK {cid} ---\n{text}\n")

    context = "\n".join(ctx_parts).strip()

    if max_chars and len(context) > max_chars:
        half = max_chars // 2
        context = context[:half] + "\n...\n" + context[-half:]

    return clean_context(context)


# ==========================
# 4. Генерация ответа (Qwen-safe)
# ==========================
def generate_rag_answer_local(
    model,
    tokenizer,
    query,
    context,
    max_new_tokens=150,
    temperature=0.0,
):
    # Очистка вопроса от опасных токенов
    query = (
        query.replace("<|im_start|>", "")
             .replace("<|im_end|>", "")
             .replace("###", "")
             .replace("User:", "")
             .replace("Assistant:", "")
             .replace("Human:", "")
    )

    # Qwen chat format
    prompt = (
        "<|im_start|>system\n"
        "You are an assistant that answers ONLY using the provided context.\n"
        "Write a short, factual answer.\n"
        "Do NOT continue the conversation.\n"
        "Do NOT ask questions.\n"
        "Do NOT add examples.\n"
        "Do NOT add disclaimers.\n"
        "Stop immediately after the answer.\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=temperature,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        pad_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(out[0], skip_special_tokens=False)

    
    answer = text.split("<|im_start|>assistant")[-1]
    answer = answer.split("<|im_end|>")[0].strip()

    return answer


# ==========================
# 5. Генерация по файлу
# ==========================
def generate_answers_for_strategy(
    retrievals_path,
    chunks_path,
    out_jsonl,
    model,
    tokenizer,
    top_k=5,
    max_context_chars=None,
    max_new_tokens=150,
    temperature=0.0
):
    chunks_map = load_chunks(chunks_path)
    os.makedirs(os.path.dirname(out_jsonl) or ".", exist_ok=True)

    with open(retrievals_path, "r", encoding="utf-8") as fin, \
         open(out_jsonl, "w", encoding="utf-8") as fout:

        for line in tqdm(fin, desc=f"Gen from {os.path.basename(retrievals_path)}"):
            item = json.loads(line)

            query = item.get("query")
            retrieved = item.get("retrieved", [])
            qid = item.get("id") or item.get("query")[:120]

            context = build_context_from_retrieved(
                retrieved,
                chunks_map,
                top_k=top_k,
                max_chars=max_context_chars
            )

            answer = generate_rag_answer_local(
                model=model,
                tokenizer=tokenizer,
                query=query,
                context=context,
                max_new_tokens=max_new_tokens,
                temperature=temperature
            )

            out = {
                "id": qid,
                "query": query,
                "retrieved": retrieved[:top_k],
                "context": context,
                "answer": answer
            }

            fout.write(json.dumps(out, ensure_ascii=False) + "\n")

    print("Generation finished, saved to", out_jsonl)


### один вопрос ###

In [ ]:
# ТЕСТ ГЕНЕРАЦИИ ДЛЯ ОДНОГО КОНКРЕТНОГО ВОПРОСА


chunks_path = "chunks/chunks_fixed_250_overlap0.1.jsonl"
retrievals_path = "results_bge_large_en_v15/results_dense_bge_large/results_dense_fixed_250_overlap0.1.jsonl"



chunks_map = load_chunks(chunks_path)


target_query_text = "how can users interact with provider models"

test_query = None
test_retrieved = None


with open(retrievals_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if target_query_text.lower() in item["query"].lower():
            test_query = item["query"]
            test_retrieved = item["retrieved"]
            break

if test_query is None:
    print("Вопрос не найден в retrieval-файле!")
else:
    print("Тестовый вопрос:", test_query)

    
    context = build_context_from_retrieved(
        test_retrieved,
        chunks_map,
        top_k=5,
        max_chars=4000
    )

    print("\n=== Контекст ===\n")
    print(context[:1500], "...\n")

    
    answer = generate_rag_answer_local(
        model=model,
        tokenizer=tokenizer,
        query=test_query,
        context=context,
        max_new_tokens=150,
        temperature=0.0
    )

    print("\n=== Ответ модели ===\n")
    print(answer)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Тестовый вопрос: how can users interact with provider models?

=== Контекст ===

--- CHUNK creating-new-model-provider.mdx_chunk_2 --- determine how users will interact with your provider's models: Predefined Models (`predefined-model`) These are models that only require unified provider credentials to use. Once a user configures their API key or other authentication details for the provider, they can immediately access all predefined models. **Example:** The `OpenAI` provider offers predefined models like `gpt-3.5-turbo-0125` and `gpt-4o-2024-05-13`. A user only needs to configure their OpenAI API key once to access all these models. Custom Models (`customizable-model`) These require additional configuration for each specific model instance. This approach is useful when models need individual parameters beyond the provider-level credentials. **Example:** `Xinference` supports both LLM and Text Embedding, but each model has a unique **model_uid**. Users must configure this model_uid se

### Запустим пакетку ###

In [ ]:
# === Автоматический запуск генерации для выбранных стратегий (BGE Large reranker) ===

BASE_RESULTS = "results_bge_large_en_v15"
OUT_DIR = "rag_outputs_bge_large"

os.makedirs(OUT_DIR, exist_ok=True)

strategies = [
    # ============================
    # DENSE
    # ============================
    {
        "name": "fixed_400_overlap0.3_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_400_overlap0.3.jsonl",
        "chunks":     "chunks/chunks_fixed_400_overlap0.3.jsonl",
    },
    {
        "name": "fixed_400_overlap0.2_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_400_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_fixed_400_overlap0.2.jsonl",
    },
    {
        "name": "fixed_50_overlap0.2_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_50_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_fixed_50_overlap0.2.jsonl",
    },
    {
        "name": "fixed_300_overlap0.1_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_300_overlap0.1.jsonl",
        "chunks":     "chunks/chunks_fixed_300_overlap0.1.jsonl",
    },
    {
        "name": "fixed_150_overlap0.2_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_150_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_fixed_150_overlap0.2.jsonl",
    },
    {
        "name": "fixed_250_overlap0.2_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_250_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_fixed_250_overlap0.2.jsonl",
    },
    {
        "name": "fixed_600_overlap0.1_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_fixed_600_overlap0.1.jsonl",
        "chunks":     "chunks/chunks_fixed_600_overlap0.1.jsonl",
    },

    # NEW structured dense
    {
        "name": "structured_400_overlap0.1_dense",
        "retrievals": f"{BASE_RESULTS}/results_dense_bge_large/results_dense_structured_400_overlap0.1.jsonl",
        "chunks":     "chunks/chunks_structured_400_overlap0.1.jsonl",
    },

    # ============================
    # HYBRID
    # ============================
    {
        "name": "fixed_400_overlap0.3_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_fixed_400_overlap0.3.jsonl",
        "chunks":     "chunks/chunks_fixed_400_overlap0.3.jsonl",
    },
    {
        "name": "fixed_600_overlap0.2_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_fixed_600_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_fixed_600_overlap0.2.jsonl",
    },

    
    {
        "name": "fixed_600_overlap0.3_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_fixed_600_overlap0.3.jsonl",
        "chunks":     "chunks/chunks_fixed_600_overlap0.3.jsonl",
    },
    {
        "name": "fixed_250_overlap0.3_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_fixed_250_overlap0.3.jsonl",
        "chunks":     "chunks/chunks_fixed_250_overlap0.3.jsonl",
    },
    {
        "name": "fixed_50_overlap0.3_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_fixed_50_overlap0.3.jsonl",
        "chunks":     "chunks/chunks_fixed_50_overlap0.3.jsonl",
    },

    
    {
        "name": "structured_600_overlap0.2_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_structured_600_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_structured_600_overlap0.2.jsonl",
    },
    {
        "name": "structured_300_overlap0.2_hybrid",
        "retrievals": f"{BASE_RESULTS}/results_hybrid_bge_large/results_hybrid_structured_300_overlap0.2.jsonl",
        "chunks":     "chunks/chunks_structured_300_overlap0.2.jsonl",
    }
]

# Добавляем путь сохранения
for s in strategies:
    s["out"] = f"{OUT_DIR}/answers_{s['name']}.jsonl"

# Запуск
for s in strategies:
    print(f"\n=== Generating for {s['name']} ===")
    generate_answers_for_strategy(
        retrievals_path=s["retrievals"],
        chunks_path=s["chunks"],
        out_jsonl=s["out"],
        model=model,
        tokenizer=tokenizer,
        top_k=5,
        max_context_chars=8000,
        max_new_tokens=180,
        temperature=0.0
    )



=== Generating for fixed_400_overlap0.3_dense ===


Gen from results_dense_fixed_400_overlap0.3.jsonl: 89it [1:26:43, 58.47s/it] 


Generation finished, saved to rag_outputs_bge_large/answers_fixed_400_overlap0.3_dense.jsonl

=== Generating for fixed_400_overlap0.2_dense ===


Gen from results_dense_fixed_400_overlap0.2.jsonl: 89it [1:25:21, 57.55s/it]


Generation finished, saved to rag_outputs_bge_large/answers_fixed_400_overlap0.2_dense.jsonl

=== Generating for fixed_50_overlap0.2_dense ===


...

Generation finished, saved to rag_outputs_bge_large/answers_fixed_50_overlap0.3_hybrid.jsonl

=== Generating for structured_600_overlap0.2_hybrid ===


Gen from results_hybrid_structured_600_overlap0.2.jsonl: 89it [1:13:39, 49.66s/it]


Generation finished, saved to rag_outputs_bge_large/answers_structured_600_overlap0.2_hybrid.jsonl

=== Generating for structured_300_overlap0.2_hybrid ===


Gen from results_hybrid_structured_300_overlap0.2.jsonl: 89it [1:09:49, 47.07s/it]

Generation finished, saved to rag_outputs_bge_large/answers_structured_300_overlap0.2_hybrid.jsonl


### Теперь посчитаем BLEU / ROUGE‑L / BERTScore ###

In [11]:
import os
import json
import csv
from glob import glob
from pathlib import Path

import nltk
nltk.download('punkt', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

from bert_score import score as bert_score
import torch

# ----------------- Параметры -----------------
GOLDEN_PATH = "golden_master.json"
ANSWERS_GLOB = "rag_outputs_bge_large/*.jsonl"
OUT_DIR = "eval_text_metrics_BLEU_ROUGEL_BERTScore_bge_large"
os.makedirs(OUT_DIR, exist_ok=True)

# ОДНА модель для всех файлов — поддерживаемая BERTScore
BERTSCORE_MODEL = "distilbert-base-uncased"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ----------------- Загрузка golden -----------------
gold_map = {}
with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
    for item in data:
        q = (item.get("query") or "").strip()
        gold = item.get("gold_answer") or item.get("expected_answer") or item.get("answer")
        if q and gold:
            gold_map[q.lower()] = gold.strip()
print(f"Loaded {len(gold_map)} golden entries from {GOLDEN_PATH}")

# ----------------- Инициализация метрик -----------------
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1

summary_rows = []

# ----------------- Основной цикл по файлам ответов -----------------
for answers_path in sorted(glob(ANSWERS_GLOB)):
    basename = os.path.basename(answers_path)
    out_csv = os.path.join(OUT_DIR, basename.replace(".jsonl", "_metrics.csv"))
    out_jsonl = os.path.join(OUT_DIR, basename.replace(".jsonl", "_metrics.jsonl"))

    raw_items = []
    answers = []
    golds = []

    # Загружаем файл с ответами
    with open(answers_path, "r", encoding="utf-8") as fin:
        for line in fin:
            item = json.loads(line)
            qid = (item.get("id") or "").strip()
            query = (item.get("query") or "").strip()
            answer = (item.get("answer") or "").strip()

            qid_norm = qid.lower()
            query_norm = query.lower()

            gold = None
            if qid_norm in gold_map:
                gold = gold_map[qid_norm]
            elif query_norm in gold_map:
                gold = gold_map[query_norm]

            raw_items.append({"id": qid or query, "query": query, "answer": answer, "gold": gold})

            if gold is not None:
                answers.append(answer)
                golds.append(gold)

    if not answers:
        print(f"No matched pairs found in {answers_path} — пропускаем.")
        continue

    # ----------------- BERTScore -----------------
    print(f"Computing BERTScore for {basename} using {BERTSCORE_MODEL}")
    P, R, F1 = bert_score(
        answers,
        golds,
        model_type=BERTSCORE_MODEL,
        lang="en",
        device=device,
        verbose=False,
        rescale_with_baseline=False
    )
    bert_f1_list = [float(x) for x in F1]

    # ----------------- BLEU/ROUGE + сбор -----------------
    rows = []
    bert_idx = 0

    for rec in raw_items:
        if rec["gold"] is None:
            rows.append({
                "id": rec["id"],
                "query": rec["query"],
                "bleu": None,
                "rougeL": None,
                "bertscore_f1": None,
                "note": "no_gold"
            })
            continue

        ans = rec["answer"]
        gold = rec["gold"]

        ref_tokens = word_tokenize(gold)
        hyp_tokens = word_tokenize(ans)

        try:
            bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth)
        except Exception:
            bleu = 0.0

        try:
            rouge_l = rouge.score(gold, ans)["rougeL"].fmeasure
        except Exception:
            rouge_l = 0.0

        bert_f1 = bert_f1_list[bert_idx]
        bert_idx += 1

        rows.append({
            "id": rec["id"],
            "query": rec["query"],
            "bleu": bleu,
            "rougeL": rouge_l,
            "bertscore_f1": bert_f1,
            "note": None
        })

    # ----------------- Сохранение -----------------
    with open(out_csv, "w", encoding="utf-8", newline="") as fcsv:
        writer = csv.writer(fcsv)
        writer.writerow(["id", "query", "bleu", "rougeL", "bertscore_f1", "note"])
        for r in rows:
            writer.writerow([r["id"], r["query"], r["bleu"], r["rougeL"], r["bertscore_f1"], r["note"]])

    with open(out_jsonl, "w", encoding="utf-8") as fout:
        for r in rows:
            fout.write(json.dumps(r, ensure_ascii=False) + "\n")

    # ----------------- Агрегаты -----------------
    bleu_vals = [r["bleu"] for r in rows if r["bleu"] is not None]
    rouge_vals = [r["rougeL"] for r in rows if r["rougeL"] is not None]
    bert_vals = [r["bertscore_f1"] for r in rows if r["bertscore_f1"] is not None]

    bleu_mean = sum(bleu_vals) / len(bleu_vals)
    rouge_mean = sum(rouge_vals) / len(rouge_vals)
    bert_mean = sum(bert_vals) / len(bert_vals)

    print(f"Evaluated {basename}: BLEU={bleu_mean:.4f}, ROUGE-L={rouge_mean:.4f}, BERT-F1={bert_mean:.4f}")


Using device: cuda
Loaded 89 golden entries from golden_master.json
Computing BERTScore for answers_fixed_150_overlap0.2_dense.jsonl using distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3559.93it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluated answers_fixed_150_overlap0.2_dense.jsonl: BLEU=0.1000, ROUGE-L=0.3329, BERT-F1=0.8400
Computing BERTScore for answers_fixed_250_overlap0.2_dense.jsonl using distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9531.64it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluated answers_fixed_250_overlap0.2_dense.jsonl: BLEU=0.1089, ROUGE-L=0.3545, BERT-F1=0.8445
Computing BERTScore for answers_fixed_250_overlap0.3_hybrid.jsonl using distilbert-base-uncased


...

Evaluated answers_structured_300_overlap0.2_hybrid.jsonl: BLEU=0.1011, ROUGE-L=0.3367, BERT-F1=0.8432
Computing BERTScore for answers_structured_400_overlap0.1_dense.jsonl using distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8465.30it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluated answers_structured_400_overlap0.1_dense.jsonl: BLEU=0.1094, ROUGE-L=0.3410, BERT-F1=0.8422
Computing BERTScore for answers_structured_600_overlap0.2_hybrid.jsonl using distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9442.80it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluated answers_structured_600_overlap0.2_hybrid.jsonl: BLEU=0.1115, ROUGE-L=0.3518, BERT-F1=0.8487


In [12]:
import os
import json
import csv
import pandas as pd
from glob import glob

METRICS_DIR = "eval_text_metrics_BLEU_ROUGEL_BERTScore_bge_large"

# Собираем все CSV-файлы
csv_files = sorted(glob(os.path.join(METRICS_DIR, "*_metrics.csv")))

rows = []

for path in csv_files:
    basename = os.path.basename(path)
    
    # имя стратегии = имя файла без "_metrics.csv"
    strategy = basename.replace("_metrics.csv", "")
    
    df = pd.read_csv(path)
    
    # фильтруем строки без gold
    df_valid = df[df["note"].isna()]
    
    if len(df_valid) == 0:
        print(f"⚠ Нет валидных пар в {basename}")
        continue
    
    bleu_mean = df_valid["bleu"].mean()
    rouge_mean = df_valid["rougeL"].mean()
    bert_mean = df_valid["bertscore_f1"].mean()
    
    rows.append({
        "strategy": strategy,
        "n_pairs": len(df_valid),
        "BLEU_mean": bleu_mean,
        "ROUGE-L_mean": rouge_mean,
        "BERTScore_mean": bert_mean
    })

# Собираем всё в один DataFrame
summary = pd.DataFrame(rows)

# Ранжируем по BERTScore (главная метрика)
summary_sorted = summary.sort_values(by="BERTScore_mean", ascending=False)

print("=== Итоговый рейтинг стратегий (по BERTScore) ===")
display(summary_sorted)

# Сохраняем
summary_path = os.path.join(METRICS_DIR, "FINAL_STRATEGY_RANKING.csv")
summary_sorted.to_csv(summary_path, index=False)

print("Сводный рейтинг сохранён в:", summary_path)


=== Итоговый рейтинг стратегий (по BERTScore) ===


,strategy,n_pairs,BLEU_mean,ROUGE-L_mean,BERTScore_mean
5,answers_fixed_400_overlap0.3_dense,89,0.114344,0.373029,0.852337
6,answers_fixed_400_overlap0.3_hybrid,89,0.117244,0.375303,0.849265
14,answers_structured_600_overlap0.2_hybrid,89,0.111510,0.351821,0.848695
4,answers_fixed_400_overlap0.2_dense,89,0.108351,0.342377,0.846627
3,answers_fixed_300_overlap0.1_dense,89,0.113639,0.344564,0.845728
9,answers_fixed_600_overlap0.1_dense,89,0.106305,0.348974,0.845599
2,answers_fixed_250_overlap0.3_hybrid,89,0.116191,0.355361,0.845316
1,answers_fixed_250_overlap0.2_dense,89,0.108930,0.354510,0.844491
12,answers_structured_300_overlap0.2_hybrid,89,0.101148,0.336713,0.843188
13,answers_structured_400_overlap0.1_dense,89,0.109423,0.340994,0.842161


Сводный рейтинг сохранён в: eval_text_metrics_BLEU_ROUGEL_BERTScore_bge_large\FINAL_STRATEGY_RANKING.csv


In [13]:
import os
import pandas as pd
from glob import glob

METRICS_DIR = "eval_text_metrics_BLEU_ROUGEL_BERTScore_bge_large"

# Собираем все CSV-файлы
csv_files = sorted(glob(os.path.join(METRICS_DIR, "*_metrics.csv")))

rows = []

for path in csv_files:
    basename = os.path.basename(path)
    
    # имя стратегии = имя файла без "_metrics.csv"
    strategy = basename.replace("_metrics.csv", "")
    
    df = pd.read_csv(path)
    
    # оставляем только строки, где есть gold
    df_valid = df[df["note"].isna()]
    
    if len(df_valid) == 0:
        print(f"⚠ Нет валидных пар в {basename}")
        continue
    
    bleu_mean = df_valid["bleu"].mean()
    rouge_mean = df_valid["rougeL"].mean()
    bert_mean = df_valid["bertscore_f1"].mean()
    
    combined = 0.7 * bert_mean + 0.2 * rouge_mean + 0.1 * bleu_mean
    
    rows.append({
        "strategy": strategy,
        "n_pairs": len(df_valid),
        "BLEU_mean": bleu_mean,
        "ROUGE-L_mean": rouge_mean,
        "BERTScore_mean": bert_mean,
        "Combined_score": combined
    })

# Собираем всё в один DataFrame
summary = pd.DataFrame(rows)

# Ранжируем по комбинированному скору
summary_sorted = summary.sort_values(by="Combined_score", ascending=False)

print("=== Итоговый комбинированный рейтинг стратегий ===")
display(summary_sorted)

# Сохраняем
summary_path = os.path.join(METRICS_DIR, "FINAL_COMBINED_RANKING.csv")
summary_sorted.to_csv(summary_path, index=False)

print("Сводный комбинированный рейтинг сохранён в:", summary_path)


=== Итоговый комбинированный рейтинг стратегий ===


,strategy,n_pairs,BLEU_mean,ROUGE-L_mean,BERTScore_mean,Combined_score
5,answers_fixed_400_overlap0.3_dense,89,0.114344,0.373029,0.852337,0.682676
6,answers_fixed_400_overlap0.3_hybrid,89,0.117244,0.375303,0.849265,0.681271
14,answers_structured_600_overlap0.2_hybrid,89,0.111510,0.351821,0.848695,0.675602
2,answers_fixed_250_overlap0.3_hybrid,89,0.116191,0.355361,0.845316,0.674412
1,answers_fixed_250_overlap0.2_dense,89,0.108930,0.354510,0.844491,0.672939
9,answers_fixed_600_overlap0.1_dense,89,0.106305,0.348974,0.845599,0.672344
3,answers_fixed_300_overlap0.1_dense,89,0.113639,0.344564,0.845728,0.672286
4,answers_fixed_400_overlap0.2_dense,89,0.108351,0.342377,0.846627,0.671950
10,answers_fixed_600_overlap0.2_hybrid,89,0.113638,0.348603,0.842026,0.670503
13,answers_structured_400_overlap0.1_dense,89,0.109423,0.340994,0.842161,0.668654


Сводный комбинированный рейтинг сохранён в: eval_text_metrics_BLEU_ROUGEL_BERTScore_bge_large\FINAL_COMBINED_RANKING.csv
